# Project 2 — Customer Behavior & Retention

**Goal.** Build the retention, churn, segmentation, and LTV view for a subscription business so growth and product teams can target the right customers with the right action.

**Inputs.** `customers.csv`, `transactions.csv` in `../data/` (synthetic — generated by `../scripts/generate_data.py`).

**Outputs.** Cleaned analytical frames + the cohort heatmap and KPI dashboard in `../dashboard/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

DATA = Path('../data')
pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
customers = pd.read_csv(DATA / 'customers.csv',     parse_dates=['signup_date'])
tx        = pd.read_csv(DATA / 'transactions.csv',  parse_dates=['transaction_date'])
tx['month']            = tx['transaction_date'].dt.to_period('M').dt.to_timestamp()
customers['cohort_month'] = customers['signup_date'].dt.to_period('M').dt.to_timestamp()
tx = tx.merge(customers[['customer_id', 'cohort_month', 'acq_channel', 'plan']],
              on='customer_id')
print(customers.shape, tx.shape)

## 1. Headline retention metrics

In [ ]:
ltv = tx.groupby('customer_id')['amount'].sum()
last_month = tx['month'].max()
kpis = {
    'Customers'          : len(customers),
    'Total revenue (USD)': tx['amount'].sum(),
    'Avg LTV (USD)'      : ltv.mean(),
    'Median LTV (USD)'   : ltv.median(),
    'MRR last month'     : tx.loc[tx['month'] == last_month, 'amount'].sum(),
    'Active last month'  : tx.loc[tx['month'] == last_month, 'customer_id'].nunique(),
}
pd.Series(kpis).to_frame('value')

## 2. Cohort retention matrix

We assign every customer to a cohort month based on their first paid event, then measure how many of that cohort were still active in each subsequent month.

In [ ]:
act = tx[['customer_id', 'cohort_month', 'month']].drop_duplicates()
act['month_index'] = (
    (act['month'].dt.year  - act['cohort_month'].dt.year) * 12
  + (act['month'].dt.month - act['cohort_month'].dt.month)
)
size = customers.groupby('cohort_month').size().rename('size')
cohort = (act.groupby(['cohort_month', 'month_index'])['customer_id']
             .nunique().reset_index(name='customers')
             .merge(size, on='cohort_month'))
cohort['retention'] = cohort['customers'] / cohort['size'] * 100
matrix = cohort.pivot(index='cohort_month', columns='month_index', values='retention')
matrix.iloc[-12:, :13].round(1)

In [ ]:
# Average retention curve across cohorts that are old enough to be measured at month m
end = tx['transaction_date'].max().to_period('M')
rows = []
for m in range(0, 19):
    eligible = [c for c in cohort['cohort_month'].unique()
                if (end - c.to_period('M')).n >= m]
    sub = cohort[(cohort['month_index'] == m) & (cohort['cohort_month'].isin(eligible))]
    if not sub.empty:
        rows.append((m, sub['retention'].mean(), len(sub)))
curve = pd.DataFrame(rows, columns=['month_index', 'avg_retention', 'cohorts'])
curve

## 3. Monthly churn

In [ ]:
active = (tx.groupby('month')['customer_id'].agg(set)
             .reset_index(name='active'))
active['prev']    = active['active'].shift(1)
active = active.dropna()
active['churned'] = active.apply(lambda r: len(r['prev'] - r['active']), axis=1)
active['base']    = active['prev'].apply(len)
active['churn_pct'] = active['churned'] / active['base'] * 100
active[['month', 'churned', 'base', 'churn_pct']].tail(12).round(2)

## 4. LTV by acquisition channel and plan

In [ ]:
by_channel = (customers.merge(ltv.rename('ltv'), on='customer_id')
                       .groupby('acq_channel')
                       .agg(customers=('customer_id', 'count'),
                            avg_ltv=('ltv', 'mean'),
                            total_rev=('ltv', 'sum'))
                       .sort_values('avg_ltv', ascending=False))
by_channel

In [ ]:
by_plan = (customers.merge(ltv.rename('ltv'), on='customer_id')
                    .groupby('plan')
                    .agg(customers=('customer_id', 'count'),
                         avg_ltv=('ltv', 'mean'))
                    .sort_values('avg_ltv', ascending=False))
by_plan

## 5. RFM-lite segmentation

Tertiles on Recency / Frequency / Monetary, then map combinations to readable segments.

In [ ]:
end_date = tx['transaction_date'].max()
rfm = (tx.groupby('customer_id')
         .agg(last_tx=('transaction_date', 'max'),
              frequency=('transaction_id', 'count'),
              monetary=('amount', 'sum'))
         .reset_index())
rfm['recency_days'] = (end_date - rfm['last_tx']).dt.days
rfm['r_score'] = pd.qcut(rfm['recency_days'], 3, labels=[1, 2, 3]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 3, labels=[3, 2, 1]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 3, labels=[3, 2, 1]).astype(int)

def label(r):
    if r.r_score == 1 and r.f_score == 1 and r.m_score == 1: return 'Champions'
    if r.r_score == 1 and r.f_score <= 2:                    return 'Promising'
    if r.r_score == 3 and r.f_score == 1:                    return 'At Risk'
    if r.r_score == 3:                                       return 'Lost'
    return 'Loyal'

rfm['segment'] = rfm.apply(label, axis=1)
rfm['segment'].value_counts()

See `../report.md` for the executive write-up.